# Venue / Scorekeeper Bias Analysis (Phase 2, Area 4)

**Goal:** Identify venues where shot recording patterns deviate significantly from the league average, separating coordinate/distance bias from event-frequency bias before either is used in model training.

**Key questions:**
1. Which venues are statistical outliers in shot frequency or coordinate distribution?
2. Is an outlier scorekeeper behavior or real hockey context? (compare away-team metrics at outlier venues vs. elsewhere)
3. How stable is venue bias across seasons?
4. What correction approach is warranted? (categorical feature vs. pre-correction vs. hierarchical model)

**Data source:** `shot_events`, `games`, `venue_bias_diagnostics`, and the shared venue-bias helpers used by `scripts/export_venue_correction_validation_from_db.py`


In [ ]:
import os
import sys
import sqlite3

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats as sp_stats

# Add src/ to path ? handle CWD being project root or notebooks/
for _candidate in [os.path.join(os.getcwd(), "src"),
                   os.path.join(os.getcwd(), "..", "src")]:
    _candidate = os.path.abspath(_candidate)
    if os.path.isdir(_candidate) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

for _candidate in [os.path.join(os.getcwd(), "scripts"),
                   os.path.join(os.getcwd(), "..", "scripts")]:
    _candidate = os.path.abspath(_candidate)
    if os.path.isdir(_candidate) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

from database import DATABASE_PATH, BLOCKED_SHOT_EVENT_TYPE
from export_venue_correction_validation_from_db import load_event_frequency_game_rows
from venue_bias import (
    ANOMALY_REAL_SCOREKEEPER_REGIME_SUPPORTED,
    PRIMARY_EVENT_FREQUENCY_GROUP,
    PRIMARY_EVENT_FREQUENCY_SCOPE,
    annotate_event_frequency_anomalies,
    compute_event_frequency_diagnostics,
    compute_paired_away_frequency_comparisons,
    primary_event_frequency_residual_z_scores,
    top_event_frequency_anomalies,
)

ASYMMETRY_CI_BOOTSTRAPS = 5_000
ASYMMETRY_RANDOM_SEED = 42
MIN_PAIRED_TEAM_SEASONS = 10
ANALYSIS_SHOT_JOIN = f"(se.shot_event_type IS NULL OR se.shot_event_type != '{BLOCKED_SHOT_EVENT_TYPE}')"


def p_value_label(p_value):
    if p_value is None:
        return "N/A"
    return "<0.0001" if p_value < 0.0001 else f"{p_value:.4f}"


def bootstrap_mean_ci(values, n_boot=ASYMMETRY_CI_BOOTSTRAPS):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return 0.0, 0.0, 0.0
    rng = np.random.default_rng(ASYMMETRY_RANDOM_SEED)
    samples = rng.choice(values, size=(n_boot, values.size), replace=True)
    means = samples.mean(axis=1)
    return float(values.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def paired_cohens_d(diffs):
    diffs = np.asarray(diffs, dtype=float)
    if diffs.size < 2:
        return None
    std = float(diffs.std(ddof=1))
    if std == 0.0:
        return 0.0
    return float(diffs.mean() / std)


sns.set_theme(style="whitegrid")
print(f"Database: {DATABASE_PATH}")
conn = sqlite3.connect(DATABASE_PATH)
conn.row_factory = sqlite3.Row
print("Connected.")


## 1. Data availability check

Verify venue metadata is populated in the `games` table and check how many seasons/venues are available.

In [ ]:
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM games")
total_games = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM games WHERE venue_name IS NOT NULL")
with_venue = cur.fetchone()[0]

cur.execute("SELECT COUNT(DISTINCT venue_name) FROM games WHERE venue_name IS NOT NULL")
distinct_venues = cur.fetchone()[0]

cur.execute("SELECT COUNT(DISTINCT season) FROM games WHERE venue_name IS NOT NULL")
distinct_seasons = cur.fetchone()[0]

cur.execute("""
    SELECT COUNT(*)
    FROM shot_events se
    JOIN games g ON se.game_id = g.game_id
    WHERE g.venue_name IS NOT NULL
""")
shots_with_venue = cur.fetchone()[0]

print(f"Total games:                     {total_games:,}")
print(f"Games with venue_name:           {with_venue:,} ({with_venue/total_games*100:.1f}%)" if total_games else "")
print(f"Distinct venues:                 {distinct_venues}")
print(f"Distinct seasons:                {distinct_seasons}")
print(f"Shot events joinable to venues:  {shots_with_venue:,}")

if with_venue == 0:
    print("\n*** No venue data. Run the scraper to populate games with venue metadata. ***")

## 2. Per-venue shot counts (league-wide view)

Rank venues by total shots recorded per game. Outlier venues may have scorekeepers who count more liberally (especially missed/blocked shots).

In [ ]:
cur.execute("""
    SELECT g.venue_name,
           COUNT(DISTINCT g.game_id) AS games,
           COUNT(se.shot_event_id)   AS total_shots,
           ROUND(CAST(COUNT(se.shot_event_id) AS REAL) / COUNT(DISTINCT g.game_id), 1) AS shots_per_game,
           ROUND(AVG(se.distance_to_goal), 1) AS avg_distance,
           ROUND(CAST(SUM(se.is_goal) AS REAL) / COUNT(se.shot_event_id), 4) AS goal_rate
    FROM games g
    JOIN shot_events se ON g.game_id = se.game_id
    WHERE g.venue_name IS NOT NULL
    GROUP BY g.venue_name
    HAVING games >= 10
    ORDER BY shots_per_game DESC
""")

venue_rows = cur.fetchall()

if venue_rows:
    names = [r[0] for r in venue_rows]
    spg = [r[3] for r in venue_rows]
    avg_d = [r[4] for r in venue_rows]
    games_ct = [r[1] for r in venue_rows]

    league_avg_spg = sum(r[2] for r in venue_rows) / sum(r[1] for r in venue_rows)

    fig, ax = plt.subplots(figsize=(14, max(6, len(names) * 0.3)))
    colors = ["#e74c3c" if abs(s - league_avg_spg) > 5 else "#3498db" for s in spg]
    y_pos = range(len(names))
    ax.barh(y_pos, spg, color=colors, alpha=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(names, fontsize=8)
    ax.set_xlabel("Shots per game")
    ax.set_title("Shots Per Game by Venue (red = >5 from league avg)")
    ax.axvline(x=league_avg_spg, color="black", linestyle="--", linewidth=1,
               label=f"League avg: {league_avg_spg:.1f}")
    ax.legend(fontsize=9)
    ax.invert_yaxis()
    fig.tight_layout()
    plt.show()

    print(f"\nLeague average shots/game: {league_avg_spg:.1f}")
    print(f"\n{'Venue':<35} {'Games':>6} {'Shots/G':>8} {'Avg Dist':>9} {'Goal Rate':>10}")
    print("-" * 72)
    for r in venue_rows[:10]:
        print(f"{r[0]:<35} {r[1]:>6} {r[3]:>8.1f} {r[4]:>9.1f} {r[5]:>10.4f}")
    if len(venue_rows) > 10:
        print(f"  ... and {len(venue_rows) - 10} more venues")
else:
    print("No venue shot data available.")

## 3. Venue bias diagnostics (z-scores)

Populate the `venue_bias_diagnostics` table and inspect outliers. Venues with |z-score| > 2 in shot count or average distance are flagged.

In [ ]:
from database import populate_venue_diagnostics

cur.execute("""
    SELECT DISTINCT g.season FROM games g
    WHERE g.venue_name IS NOT NULL AND g.season IS NOT NULL
    ORDER BY g.season
""")
seasons = [r[0] for r in cur.fetchall()]
print(f"Seasons with venue data: {len(seasons)}")

for season in seasons:
    populate_venue_diagnostics(conn, str(season))
print("Diagnostics populated.")

cur.execute("""
    SELECT venue_name, season, total_shots, avg_distance,
           shot_count_z_score, distance_z_score, bias_flag
    FROM venue_bias_diagnostics
    WHERE bias_flag = 1
    ORDER BY ABS(shot_count_z_score) DESC
""")
flagged = cur.fetchall()

if flagged:
    print(f"\nFlagged venues (|z| > 2): {len(flagged)} venue-seasons")
    print(f"\n{'Venue':<35} {'Season':<10} {'Shots':>8} {'Avg Dist':>9} {'Shot Z':>8} {'Dist Z':>8}")
    print("-" * 82)
    for r in flagged:
        sc_z = f"{r[4]:.2f}" if r[4] is not None else "N/A"
        d_z = f"{r[5]:.2f}" if r[5] is not None else "N/A"
        print(f"{r[0]:<35} {str(r[1]):<10} {r[2]:>8,} {r[3]:>9.1f} {sc_z:>8} {d_z:>8}")
else:
    print("\nNo venues flagged as outliers.")

## 4. Z-score scatter plot (shot count vs. distance)

Visualize all venue-seasons in z-score space to identify clusters and outliers.

In [ ]:
cur.execute("""
    SELECT venue_name, season, shot_count_z_score, distance_z_score, bias_flag
    FROM venue_bias_diagnostics
    WHERE shot_count_z_score IS NOT NULL AND distance_z_score IS NOT NULL
""")
diag_rows = cur.fetchall()

if diag_rows:
    names = [r[0] for r in diag_rows]
    sc_z = np.array([r[2] for r in diag_rows])
    d_z = np.array([r[3] for r in diag_rows])
    flags = [r[4] for r in diag_rows]

    fig, ax = plt.subplots(figsize=(10, 8))

    normal = [i for i, f in enumerate(flags) if f == 0]
    flagged_idx = [i for i, f in enumerate(flags) if f == 1]

    ax.scatter(sc_z[normal], d_z[normal], alpha=0.4, s=20, color="#3498db", label="Normal")
    ax.scatter(sc_z[flagged_idx], d_z[flagged_idx], alpha=0.8, s=50, color="#e74c3c",
               marker="^", label="Flagged (|z| > 2)")

    for i in flagged_idx:
        ax.annotate(f"{names[i]}", (sc_z[i], d_z[i]),
                    fontsize=7, alpha=0.8, xytext=(5, 5),
                    textcoords="offset points")

    THRESHOLD = 2.0
    ax.axvline(x=THRESHOLD, color="#e74c3c", linestyle=":", alpha=0.5)
    ax.axvline(x=-THRESHOLD, color="#e74c3c", linestyle=":", alpha=0.5)
    ax.axhline(y=THRESHOLD, color="#e74c3c", linestyle=":", alpha=0.5)
    ax.axhline(y=-THRESHOLD, color="#e74c3c", linestyle=":", alpha=0.5)

    ax.set_xlabel("Shot count z-score")
    ax.set_ylabel("Avg distance z-score")
    ax.set_title("Venue Bias: Shot Count vs. Distance Z-Scores")
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("No diagnostics data available.")

## 5. Home-away asymmetry paired test

The key diagnostic: is a venue's high/low shot count caused by the scorekeeper, or by the home team's actual playing style? If visiting teams show inflated counts at a venue relative to their own same-season away games elsewhere, that supports a scorekeeper-bias interpretation.

For each flagged venue, compare away teams' shot counts per game **at that venue** vs. **the same away team-season's road games elsewhere**. The table reports paired mean difference, a 95% bootstrap CI, a Wilcoxon signed-rank p-value, paired Cohen's d, and a sample-size flag.


In [ ]:
cur.execute("""
    SELECT DISTINCT venue_name FROM venue_bias_diagnostics WHERE bias_flag = 1
""")
flagged_venues = [r[0] for r in cur.fetchall()]

if flagged_venues:
    results = []
    for venue in flagged_venues:
        cur.execute(f"""
            WITH away_game_shots AS (
                SELECT g.game_id,
                       g.away_team_id,
                       g.season,
                       g.venue_name,
                       COUNT(se.shot_event_id) AS away_shots
                FROM games g
                LEFT JOIN shot_events se
                  ON se.game_id = g.game_id
                 AND se.shooting_team_id = g.away_team_id
                 AND {ANALYSIS_SHOT_JOIN}
                WHERE g.away_team_id IS NOT NULL
                  AND g.season IS NOT NULL
                  AND g.venue_name IS NOT NULL
                GROUP BY g.game_id, g.away_team_id, g.season, g.venue_name
            ),
            flagged_team_seasons AS (
                SELECT DISTINCT away_team_id, season
                FROM away_game_shots
                WHERE venue_name = ?
            )
            SELECT f.away_team_id,
                   f.season,
                   AVG(CASE WHEN a.venue_name = ? THEN a.away_shots END) AS at_venue,
                   AVG(CASE WHEN a.venue_name != ? THEN a.away_shots END) AS elsewhere,
                   SUM(CASE WHEN a.venue_name = ? THEN 1 ELSE 0 END) AS at_games,
                   SUM(CASE WHEN a.venue_name != ? THEN 1 ELSE 0 END) AS elsewhere_games
            FROM flagged_team_seasons f
            JOIN away_game_shots a
              ON a.away_team_id = f.away_team_id
             AND a.season = f.season
            GROUP BY f.away_team_id, f.season
            HAVING at_games > 0 AND elsewhere_games > 0
        """, (venue, venue, venue, venue, venue))
        paired_rows = cur.fetchall()

        diffs = np.array([
            row["at_venue"] - row["elsewhere"]
            for row in paired_rows
            if row["at_venue"] is not None and row["elsewhere"] is not None
        ], dtype=float)

        if diffs.size > 0:
            mean_diff, ci_low, ci_high = bootstrap_mean_ci(diffs)
            mean_at = float(np.mean([row["at_venue"] for row in paired_rows]))
            mean_elsewhere = float(np.mean([row["elsewhere"] for row in paired_rows]))
            total_at_games = int(sum(row["at_games"] for row in paired_rows))
            total_elsewhere_games = int(sum(row["elsewhere_games"] for row in paired_rows))
        else:
            mean_diff, ci_low, ci_high = None, None, None
            mean_at, mean_elsewhere = None, None
            total_at_games, total_elsewhere_games = 0, 0

        if diffs.size >= 2 and not np.allclose(diffs, 0.0):
            _, p_value = sp_stats.wilcoxon(diffs)
        else:
            p_value = None

        effect_size = paired_cohens_d(diffs)
        underpowered = diffs.size < MIN_PAIRED_TEAM_SEASONS
        results.append({
            "venue": venue,
            "pairs": int(diffs.size),
            "at_games": total_at_games,
            "elsewhere_games": total_elsewhere_games,
            "at_venue": mean_at,
            "elsewhere": mean_elsewhere,
            "diff": mean_diff,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "p_value": p_value,
            "effect_size": effect_size,
            "underpowered": underpowered,
        })

    print(
        f"{'Venue':<32} {'Pairs':>5} {'AtG':>5} {'ElseG':>6} "
        f"{'At':>7} {'Else':>7} {'Diff':>7} {'95% CI':>19} {'p':>8} {'d':>7} {'Power':>7}"
    )
    print("-" * 124)
    for row in results:
        if row["diff"] is None:
            print(f"{row['venue']:<32} {'0':>5} {'0':>5} {'0':>6} {'N/A':>7} {'N/A':>7} {'N/A':>7} {'N/A':>19} {'N/A':>8} {'N/A':>7} {'LOW':>7}")
            continue
        ci_label = f"[{row['ci_low']:+.2f}, {row['ci_high']:+.2f}]"
        effect_label = "N/A" if row["effect_size"] is None else f"{row['effect_size']:+.2f}"
        power = "LOW" if row["underpowered"] else "OK"
        print(
            f"{row['venue']:<32} {row['pairs']:>5} {row['at_games']:>5} {row['elsewhere_games']:>6} "
            f"{row['at_venue']:>7.1f} {row['elsewhere']:>7.1f} {row['diff']:>+7.2f} "
            f"{ci_label:>19} {p_value_label(row['p_value']):>8} {effect_label:>7} {power:>7}"
        )

    print("\nPositive diff = same away team-seasons record MORE shots per game at this venue than elsewhere.")
    print("Evidence is strongest when the CI excludes 0, p < 0.05, |d| is practically meaningful, and power is OK.")
else:
    print("No flagged venues to analyze.")


## 6. Coordinate distribution by venue

Do some venues show systematically different x/y coordinate distributions? This can indicate that the scorekeeper is guessing shot locations differently.

In [ ]:
cur.execute("""
    SELECT venue_name, x_coord_mean, x_coord_stddev, y_coord_mean, y_coord_stddev,
           total_shots
    FROM venue_bias_diagnostics
    WHERE x_coord_mean IS NOT NULL
    ORDER BY total_shots DESC
""")
coord_rows = cur.fetchall()

if coord_rows:
    v_names = [r[0] for r in coord_rows]
    x_std = np.array([r[2] for r in coord_rows])
    y_std = np.array([r[4] for r in coord_rows])
    sizes = np.array([r[5] for r in coord_rows])

    size_norm = 20 + 80 * (sizes - sizes.min()) / (sizes.max() - sizes.min() + 1)

    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(x_std, y_std, s=size_norm, alpha=0.5, color="#3498db")

    x_thresh_hi = np.percentile(x_std, 95)
    x_thresh_lo = np.percentile(x_std, 5)
    y_thresh_hi = np.percentile(y_std, 95)
    y_thresh_lo = np.percentile(y_std, 5)

    for i, name in enumerate(v_names):
        if x_std[i] > x_thresh_hi or x_std[i] < x_thresh_lo or \
           y_std[i] > y_thresh_hi or y_std[i] < y_thresh_lo:
            ax.annotate(name, (x_std[i], y_std[i]), fontsize=7, alpha=0.8,
                        xytext=(5, 5), textcoords="offset points")

    ax.set_xlabel("X-coordinate std dev")
    ax.set_ylabel("Y-coordinate std dev")
    ax.set_title("Shot Coordinate Spread by Venue-Season (size = shot count)")
    fig.tight_layout()
    plt.show()
else:
    print("No coordinate diagnostics available.")

## 7. Seasonal stability of venue bias

Is bias stable across seasons (persistent scorekeeper) or volatile (random noise, new scorekeeper, arena renovation)? For venues with multiple seasons of data, plot z-score trends.

In [ ]:
cur.execute("""
    SELECT venue_name, COUNT(DISTINCT season) AS n_seasons,
           SUM(bias_flag) AS times_flagged
    FROM venue_bias_diagnostics
    GROUP BY venue_name
    HAVING n_seasons >= 3
    ORDER BY times_flagged DESC
""")
multi_season_venues = cur.fetchall()

MAX_VENUES_TO_PLOT = 8
venues_to_plot = [r[0] for r in multi_season_venues[:MAX_VENUES_TO_PLOT]]

if venues_to_plot:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

    for venue in venues_to_plot:
        cur.execute("""
            SELECT season, shot_count_z_score, distance_z_score
            FROM venue_bias_diagnostics
            WHERE venue_name = ?
            ORDER BY season
        """, (venue,))
        rows = cur.fetchall()
        if not rows:
            continue

        s_seasons = [str(r[0]) for r in rows]
        s_sc_z = [r[1] for r in rows]
        s_d_z = [r[2] for r in rows]

        ax1.plot(s_seasons, s_sc_z, marker="o", markersize=4, linewidth=1.2,
                 alpha=0.7, label=venue[:25])
        ax2.plot(s_seasons, s_d_z, marker="o", markersize=4, linewidth=1.2,
                 alpha=0.7, label=venue[:25])

    ax1.axhline(y=0, color="black", linewidth=0.5)
    ax1.axhline(y=2, color="#e74c3c", linestyle=":", alpha=0.5)
    ax1.axhline(y=-2, color="#e74c3c", linestyle=":", alpha=0.5)
    ax1.set_ylabel("Shot count z-score")
    ax1.set_title("Shot Count Z-Score Across Seasons")
    ax1.legend(fontsize=7, loc="upper left", ncol=2)

    ax2.axhline(y=0, color="black", linewidth=0.5)
    ax2.axhline(y=2, color="#e74c3c", linestyle=":", alpha=0.5)
    ax2.axhline(y=-2, color="#e74c3c", linestyle=":", alpha=0.5)
    ax2.set_ylabel("Distance z-score")
    ax2.set_xlabel("Season")
    ax2.set_title("Avg Distance Z-Score Across Seasons")
    ax2.legend(fontsize=7, loc="upper left", ncol=2)

    plt.xticks(rotation=45, fontsize=8)
    fig.tight_layout()
    plt.show()

    print("Stable lines near +/-2 = persistent bias (same scorekeeper pattern)")
    print("Volatile swings = noise or personnel changes")
else:
    print("Not enough multi-season venue data to analyze stability.")

## 8. Event-frequency scorekeeper diagnostics

Distance/location bias and event-frequency bias are separate scorer behaviors. The primary frequency gate uses regular-season training attempts, because schedule exposure is most comparable there. Blocked-shot and all-attempt frequencies are retained as diagnostics, but blocked shots remain outside the current shot-level xG training contract.


In [ ]:
frequency_game_rows = load_event_frequency_game_rows(conn)
frequency_diagnostics = compute_event_frequency_diagnostics(frequency_game_rows)
paired_frequency = compute_paired_away_frequency_comparisons(frequency_game_rows)
annotated_frequency = annotate_event_frequency_anomalies(
    frequency_diagnostics, paired_frequency
)
primary_frequency_residuals = primary_event_frequency_residual_z_scores(
    annotated_frequency
)
top_frequency_anomalies = top_event_frequency_anomalies(
    annotated_frequency, limit=15
)

frequency_candidates = [
    row for row in annotated_frequency if row.get("candidate_anomaly")
]
supported_frequency = [
    row for row in frequency_candidates
    if row["anomaly_classification"] == ANOMALY_REAL_SCOREKEEPER_REGIME_SUPPORTED
]

print(f"Frequency game/group/scope rows: {len(frequency_game_rows):,}")
print(f"Primary frequency gate: {PRIMARY_EVENT_FREQUENCY_SCOPE}:{PRIMARY_EVENT_FREQUENCY_GROUP}")
print(f"Primary residual venue-seasons: {len(primary_frequency_residuals):,}")
print(f"Candidate frequency anomalies: {len(frequency_candidates):,}")
print(f"Supported real-scorekeeper regimes: {len(supported_frequency):,}")

if top_frequency_anomalies:
    print("\nTop event-frequency anomalies:")
    print(f"{'Scope':<18} {'Group':<18} {'Venue-season':<45} {'z':>7} {'Events/g':>9} {'Paired d':>9} {'Class':<34} {'Known':<5}")
    print("-" * 155)
    for row in top_frequency_anomalies:
        paired_d = row.get("paired_cohens_d")
        paired_d_text = f"{paired_d:.2f}" if paired_d is not None else "N/A"
        known = "YES" if row.get("known_scorekeeper_prior") else "NO"
        venue_season = f"{row['season']}:{row['venue_name']}"
        print(
            f"{row['game_type_scope']:<18} "
            f"{row['event_group']:<18} "
            f"{venue_season:<45} "
            f"{row['frequency_z_score']:>7.2f} "
            f"{row['events_per_game']:>9.2f} "
            f"{paired_d_text:>9} "
            f"{row['anomaly_classification']:<34} "
            f"{known:<5}"
        )
else:
    print("No event-frequency anomalies exceeded the reporting threshold.")


## 9. Summary and recommendations

**Interpret the results above to decide:**

1. **Which venues need correction?** Venues with |z-score| > 2 are candidates only if the paired away-team/season asymmetry test supports a non-zero effect with adequate sample size.

2. **Separate correction families:** Distance/location residuals and event-frequency residuals must be evaluated separately. A venue can require one correction family without requiring the other.

3. **Event-frequency policy:** Regular-season training-attempt frequency is the primary gate. Blocked-shot and all-attempt frequencies are diagnostic until a later model explicitly uses those event families.

4. **Correction approach:**
   - If bias is small and unstable, keep venue as an exploratory categorical feature only.
   - If bias is moderate and the paired test supports it, validate a prior-season venue feature in `model_validation_framework.ipynb`.
   - If bias is large and stable, consider hierarchical partial pooling, but still validate on held-out seasons.

5. **Seasonal stability:** If a venue's bias is stable across 3+ seasons, use prior-season estimates for held-out validation. Do not use same-season diagnostics as model features.

6. **Limitation:** The NHL API does not provide scorekeeper identity. Bias is venue-level only, which limits decomposition between scorekeeper change vs. arena renovation effects.


In [ ]:
conn.close()
print("Done.")